# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I selected Logistic Regression, Decision Tree, and Random Forest because my lane is Refresh / Content Opportunity Scoring. The underlying task is supervised classification: the model learns whether a content page is associated with the observed declining label, and the predicted probability can then be used to rank pages for review.

Logistic Regression provides a simple and interpretable learned model. Decision Tree can capture non-linear relationships and readable decision rules. Random Forest can capture more complex interactions while reducing the instability of a single tree.

I will not choose a model because it is more complex. I will compare all models against my Week-4 hand-written baseline using the same held-out clients and the same ranking metrics, especially Precision@50 and average precision.


## 2. Split design

I use a client-grouped holdout split. The dataset contains multiple content pages belonging to the same client, so randomly splitting individual rows could place pages from the same client in both the training and test sets.

Instead, I hold out complete clients for testing. This means the model is evaluated on pages belonging to clients it did not see during training.

This is a more honest validation design for this dataset because it reduces the risk that the model learns client-specific patterns that make the test result look artificially strong.

I will use the same held-out test set for the baseline and all three ML models so that the comparison is fair.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
# Load starter dataset
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/hafizahmadadilaiengineer/flyrank-ml-internship.git

repo_root = Path("flyrank-ml-internship")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")



print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())
print("Unique content pages:", df["content_id"].nunique())

Dataset shape: (30000, 44)
Unique clients: 32
Unique content pages: 30000


In [26]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Declining pages:", df["is_declining_label"].sum())
print("Non-declining pages:", (df["is_declining_label"] == 0).sum())

Declining pages: 16262
Non-declining pages: 13738


## 3. Train + compare vs my baseline

I will train Logistic Regression, Decision Tree, and Random Forest models using the same starter dataset and the same client-grouped train/test split defined above.

The target is `is_declining_label`, which is a proxy label derived from the observed `trend_direction`. The model will use safe observable features and will exclude `trend_direction`, `trend_pct`, `content_id`, and `client_id`.

Each model will produce a probability of the declining class. I will use that probability to rank pages for review and compare the resulting ranking against my Week-4 hand-written baseline.

The primary ranking metric is Precision@50 because the business decision is to identify the most useful pages for a limited review queue. Average precision will also be reported to evaluate the quality of the complete ranking.

All models and the baseline will be evaluated on exactly the same held-out clients.

In [27]:
from sklearn.model_selection import train_test_split

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))

Training clients: 25
Test clients: 7

Training rows: 26581
Test rows: 3419


In [28]:
overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

Client overlap: 0


In [29]:
print(
    "Train declining rate:",
    train_df["is_declining_label"].mean()
)

print(
    "Test declining rate:",
    test_df["is_declining_label"].mean()
)

Train declining rate: 0.5444114216921861
Test declining rate: 0.5238373793506873


In [30]:
import numpy as np
import pandas as pd

# Target
target = "is_declining_label"

# Safe numeric features from the starter model
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Safe categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Explicitly exclude leakage/context columns
excluded_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Excluded:", excluded_features)

Numeric features: 18
Categorical features: 8
Excluded: ['content_id', 'client_id', 'trend_direction', 'trend_pct']


In [31]:
X_train = train_df[numeric_features + categorical_features].copy()
X_test = test_df[numeric_features + categorical_features].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (26581, 26)
X_test: (3419, 26)
y_train: (26581,)
y_test: (3419,)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

In [34]:
from sklearn.pipeline import Pipeline

trained_models = {}
predictions = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    prob = pipeline.predict_proba(X_test)[:, 1]

    trained_models[name] = pipeline
    predictions[name] = prob

    print(f"{name} trained successfully.")

Logistic Regression trained successfully.
Decision Tree trained successfully.
Random Forest trained successfully.


In [35]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

In [36]:
from sklearn.metrics import average_precision_score

results = []

for name, scores in predictions.items():

    results.append({
        "Method": name,
        "Precision@50": precision_at_k(
            y_test.reset_index(drop=True),
            pd.Series(scores),
            k=50
        ),
        "Average Precision": average_precision_score(
            y_test,
            scores
        )
    })

In [37]:
baseline_test = test_df.copy()

baseline_test["baseline_score"] = 0

baseline_test.loc[
    baseline_test["days_since_last_update"] >= 180,
    "baseline_score"
] += 40

baseline_test.loc[
    baseline_test["impressions_90d"] >= 500,
    "baseline_score"
] += 30

baseline_test.loc[
    (baseline_test["avg_position"] > 0) &
    (baseline_test["avg_position"] <= 20),
    "baseline_score"
] += 20

baseline_test.loc[
    baseline_test["ctr"] < 0.5,
    "baseline_score"
] += 10

In [38]:
baseline_precision_50 = precision_at_k(
    baseline_test[target].reset_index(drop=True),
    baseline_test["baseline_score"].reset_index(drop=True),
    k=50
)

baseline_ap = average_precision_score(
    baseline_test[target],
    baseline_test["baseline_score"]
)

results.append({
    "Method": "Week-4 Baseline",
    "Precision@50": baseline_precision_50,
    "Average Precision": baseline_ap
})

In [39]:
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "Method",
        "Precision@50",
        "Average Precision"
    ]
].sort_values(
    "Precision@50",
    ascending=False
)

display(results_df)

,Method,Precision@50,Average Precision
3,Week-4 Baseline,0.72,0.571718
0,Logistic Regression,0.62,0.599466
2,Random Forest,0.46,0.639209
1,Decision Tree,0.42,0.610575


## 4. Errors and Interpretation

The Random Forest model achieved 0.81 recall for the declining class, meaning it identified a large proportion of the observed declining pages. However, it also produced many false positives: 932 non-declining pages were classified as declining.

For the ranked queue, Random Forest achieved Precision@50 of 0.46, meaning 23 of the top 50 recommended pages were actually declining according to the observed label. This was lower than the Week-4 baseline Precision@50 of 0.72.

The false-positive examples show pages with strong search visibility but very low CTR being assigned high model scores even though their observed declining label is 0. All ten examples inspected were keyword articles. This suggests that the model can prioritize visible, low-CTR pages even when the observed outcome does not indicate decline. However, the inspected examples are not sufficient to claim that content type itself causes the error.

The false-negative examples show the opposite pattern. Several declining pages received very low model scores. In the examples inspected, these pages generally had extremely low impression counts, often between 1 and 4 impressions over 90 days. This suggests that the model may have difficulty identifying some declining pages with very little search activity.

Overall, the model shows a trade-off between identifying more declining pages and maintaining precision in the highest-priority queue. Random Forest achieved the highest Average Precision (0.6392), but it did not outperform the Week-4 baseline on Precision@50. Therefore, I would not replace the baseline with Random Forest for the top-50 review decision based on this experiment alone.

In [40]:
from sklearn.metrics import confusion_matrix, classification_report

rf_pipeline = trained_models["Random Forest"]

rf_pred = rf_pipeline.predict(X_test)

print("Random Forest confusion matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification report:")
print(classification_report(y_test, rf_pred))

Random Forest confusion matrix:
[[ 696  932]
 [ 332 1459]]

Classification report:
              precision    recall  f1-score   support

           0       0.68      0.43      0.52      1628
           1       0.61      0.81      0.70      1791

    accuracy                           0.63      3419
   macro avg       0.64      0.62      0.61      3419
weighted avg       0.64      0.63      0.62      3419



In [41]:
rf_scores = predictions["Random Forest"]

rf_top50 = test_df.copy()

rf_top50["model_score"] = rf_scores

rf_top50 = (
    rf_top50
    .sort_values("model_score", ascending=False)
    .head(50)
)

print("Random Forest Top 50")
print(
    "Declining pages:",
    rf_top50["is_declining_label"].sum()
)

print(
    "Precision@50:",
    rf_top50["is_declining_label"].mean()
)

Random Forest Top 50
Declining pages: 23
Precision@50: 0.46


In [42]:
rf_false_positives = rf_top50[
    rf_top50["is_declining_label"] == 0
]

display(
    rf_false_positives[
        [
            "content_id",
            "is_declining_label",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10)
)

,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update
22042,content_2ba626fea4d6,0,0.861694,360,0.00,7.2,104
5011,content_c148e44db30d,0,0.861307,335,0.00,31.3,104
10080,content_35d63627bf3e,0,0.860198,1525,0.00,32.6,103
11061,content_0b47dae0c7f9,0,0.859965,1191,0.00,23.1,103
22524,content_846bb4dd8b44,0,0.858742,870,0.11,17.6,104
13,content_a5a2fbc76336,0,0.852345,307,0.00,39.8,103
5477,content_3164f3076003,0,0.851773,2696,0.04,16.1,104
28718,content_ef6e7d7cfe15,0,0.846040,264,0.00,22.2,104
2357,content_8f1409b2674e,0,0.845996,209,0.00,20.0,104
18475,content_197a5b1ed096,0,0.843173,605,0.00,14.4,102


In [43]:
rf_all = test_df.copy()

rf_all["model_score"] = rf_scores

rf_false_negatives = (
    rf_all[
        rf_all["is_declining_label"] == 1
    ]
    .sort_values("model_score")
)

display(
    rf_false_negatives[
        [
            "content_id",
            "is_declining_label",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update"
        ]
    ].head(10)
)

,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update
12364,content_3a4e24a3a6a8,1,0.054604,1,0.0,4.0,20
18171,content_24796d98b025,1,0.081514,1,0.0,2.0,20
22116,content_f83003a037ad,1,0.082600,2,0.0,33.0,20
4524,content_1c32d46552ba,1,0.090250,2,0.0,70.0,20
21507,content_1989eb00963b,1,0.119968,4,0.0,7.5,20
19104,content_a860ee9e7ae4,1,0.125234,1,0.0,8.0,104
22285,content_f85aa6e9bc6e,1,0.130050,2,0.0,21.5,20
12114,content_47d62e68cd7a,1,0.134275,2,0.0,20.0,20
27395,content_e72e6c56f0a3,1,0.138401,1,0.0,9.0,20
18423,content_77e2a54525b6,1,0.140505,1,0.0,7.0,20


In [44]:
comparison = test_df.copy()

comparison["baseline_score"] = baseline_test["baseline_score"].values
comparison["rf_score"] = rf_scores

comparison["baseline_rank"] = (
    comparison["baseline_score"]
    .rank(method="first", ascending=False)
)

comparison["rf_rank"] = (
    comparison["rf_score"]
    .rank(method="first", ascending=False)
)

comparison["rank_difference"] = (
    comparison["baseline_rank"] -
    comparison["rf_rank"]
)

display(
    comparison[
        [
            "content_id",
            "is_declining_label",
            "baseline_score",
            "rf_score",
            "baseline_rank",
            "rf_rank",
            "rank_difference"
        ]
    ]
    .sort_values("rank_difference", ascending=False)
    .head(10)
)

,content_id,is_declining_label,baseline_score,rf_score,baseline_rank,rf_rank,rank_difference
14532,content_29ffa7b58884,0,0,0.813647,3366.0,63.0,3303.0
7065,content_fd1c9be3a841,0,0,0.815616,3340.0,59.0,3281.0
28718,content_ef6e7d7cfe15,0,10,0.846040,3272.0,15.0,3257.0
29930,content_52a0cf81179f,1,10,0.757886,3308.0,163.0,3145.0
27820,content_2e5461c6fe96,0,10,0.788652,3233.0,95.0,3138.0
27832,content_f68bcc71c8f5,0,10,0.782870,3235.0,103.0,3132.0
27822,content_ce17b2af6bde,1,10,0.763022,3234.0,145.0,3089.0
24278,content_0f256e5b0b88,0,10,0.823158,3129.0,49.0,3080.0
28469,content_8a234f7d62b2,0,10,0.751428,3258.0,192.0,3066.0
28366,content_d1e879090911,1,10,0.752387,3251.0,187.0,3064.0


In [45]:
rf_scores = predictions["Random Forest"]

rf_top50 = test_df.copy()
rf_top50["model_score"] = rf_scores

rf_top50 = (
    rf_top50
    .sort_values("model_score", ascending=False)
    .head(50)
)

print("Random Forest Top 50")
print("Declining pages:", rf_top50["is_declining_label"].sum())
print("Precision@50:", rf_top50["is_declining_label"].mean())

Random Forest Top 50
Declining pages: 23
Precision@50: 0.46


In [46]:
rf_false_positives = rf_top50[
    rf_top50["is_declining_label"] == 0
]

display(
    rf_false_positives[
        [
            "content_id",
            "is_declining_label",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
            "content_type"
        ]
    ].head(10)
)

,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update,content_type
22042,content_2ba626fea4d6,0,0.861694,360,0.00,7.2,104,keyword article
5011,content_c148e44db30d,0,0.861307,335,0.00,31.3,104,keyword article
10080,content_35d63627bf3e,0,0.860198,1525,0.00,32.6,103,keyword article
11061,content_0b47dae0c7f9,0,0.859965,1191,0.00,23.1,103,keyword article
22524,content_846bb4dd8b44,0,0.858742,870,0.11,17.6,104,keyword article
13,content_a5a2fbc76336,0,0.852345,307,0.00,39.8,103,keyword article
5477,content_3164f3076003,0,0.851773,2696,0.04,16.1,104,keyword article
28718,content_ef6e7d7cfe15,0,0.846040,264,0.00,22.2,104,keyword article
2357,content_8f1409b2674e,0,0.845996,209,0.00,20.0,104,keyword article
18475,content_197a5b1ed096,0,0.843173,605,0.00,14.4,102,keyword article


In [47]:
rf_all = test_df.copy()
rf_all["model_score"] = rf_scores

rf_false_negatives = (
    rf_all[
        rf_all["is_declining_label"] == 1
    ]
    .sort_values("model_score")
)

display(
    rf_false_negatives[
        [
            "content_id",
            "is_declining_label",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "days_since_last_update",
            "content_type"
        ]
    ].head(10)
)

,content_id,is_declining_label,model_score,impressions_90d,ctr,avg_position,days_since_last_update,content_type
12364,content_3a4e24a3a6a8,1,0.054604,1,0.0,4.0,20,keyword article
18171,content_24796d98b025,1,0.081514,1,0.0,2.0,20,keyword article
22116,content_f83003a037ad,1,0.082600,2,0.0,33.0,20,keyword article
4524,content_1c32d46552ba,1,0.090250,2,0.0,70.0,20,keyword article
21507,content_1989eb00963b,1,0.119968,4,0.0,7.5,20,keyword article
19104,content_a860ee9e7ae4,1,0.125234,1,0.0,8.0,104,keyword article
22285,content_f85aa6e9bc6e,1,0.130050,2,0.0,21.5,20,keyword article
12114,content_47d62e68cd7a,1,0.134275,2,0.0,20.0,20,keyword article
27395,content_e72e6c56f0a3,1,0.138401,1,0.0,9.0,20,keyword article
18423,content_77e2a54525b6,1,0.140505,1,0.0,7.0,20,keyword article


## 5. Self-check

- [x] I selected models appropriate for my Refresh / Content Opportunity Scoring lane.
- [x] I explained why Logistic Regression, Decision Tree, and Random Forest were compared.
- [x] I used a client-grouped train/test split.
- [x] There were zero clients shared between training and test sets.
- [x] The target was based on the observed declining outcome.
- [x] I excluded trend-derived fields from the feature set.
- [x] I evaluated the baseline and ML models on the same held-out test clients.
- [x] I used Precision@50 as the main ranked-queue metric.
- [x] I also reported Average Precision.
- [x] I compared all three ML models with the Week-4 baseline.
- [x] I inspected false positives and false negatives.
- [x] I did not select a model based on complexity alone.
- [x] I used careful language when interpreting the observed errors.
- [x] I did not claim causal impact or make claims about Google's ranking algorithm.

### Model selection conclusion

On this client-held-out test split, the Week-4 baseline achieved the highest Precision@50 (0.72), while Random Forest achieved the highest Average Precision (0.6392). Because the primary business decision is to prioritize a limited top-50 review queue, the baseline remains the stronger method for that decision in this experiment. The ML models provide useful alternative rankings, but the results do not justify replacing the simpler baseline based on this evaluation alone.